In [1]:
import os, sys

notebook_dir = "/code/beh_ephys_analysis"
if notebook_dir not in sys.path:
    sys.path.insert(0, notebook_dir)

In [2]:
from behavior_and_time_alignment import beh_and_time_alignment
from session_preprocessing import ephys_opto_preprocessing, session_crosscorr, ephys_opto_crosscorr
from opto_tagging import opto_plotting_session, opto_plotting_unit
from opto_waveforms_preprocessing import opto_wf_preprocessing, re_filter_opto_waveforms
from drift_analysis import plot_session_opto_drift
import matplotlib.pyplot as plt
from utils.beh_functions import session_dirs, get_unit_tbl
from opto_sigs import cal_opto_sigs
import numpy as np
import pandas as pd
import os
from utils.plot_utils import combine_pdf_big
import spikeinterface as si
from joblib import Parallel, delayed
import pickle

In [3]:
#%pip install --force-reinstall "hdmf<5" "hdmf-zarr==0.11.0" "pynwb==2.8.3"
import hdmf, pynwb, hdmf_zarr
[hdmf.__version__, pynwb.__version__, hdmf_zarr.__version__]

['4.3.1', '2.8.3', '0.11.0']

In [3]:
# # combine opto pdfs
# data_type = 'curated'
# def process(session, data_type):
#     pdf_dir = session_dirs(session)[f'opto_dir_fig_{data_type}']
#     output_pdf = os.path.join(session_dirs(session)[f'opto_dir_{data_type}'], f'{session}_opto_tagging_png.pdf')
#     if os.path.exists(pdf_dir) and not os.path.exists(output_pdf):
#         print(session)
#         combine_pdf_big(pdf_dir, output_pdf)
# Parallel(n_jobs=4)(delayed(process)(session, data_type) for session in session_list[-14:-3])
# # for session in session_list[-14:-3]:
# #     try:
# #         process(session, data_type)
# #         plt.close('all')
# #     except:
# #         print(f'Failed to process {session}')

In [5]:
session = 'behavior_843660_2026-05-05_10-15-53'
session_dir = session_dirs(session)

In [ ]:
session_dir

In [4]:
unit_tbl = get_unit_tbl('behavior_808650_2025-09-24_12-38-00','raw',summary=True)
unit_tbl.head()

,unit_id,bl_max_p,p_max,p_mean,lat_max_p,lat_mean,euc_max_p,corr_max_p,opto_pass,amp,...,mat_wf_raw_fake,mat_wf_raw_fake_aligned,peak_waveform_raw_fake,peak_waveform_raw_fake_aligned,amplitude_raw_fake,peak_raw_fake,x_ccf,y_ccf,z_ccf,loc_along_probe
0,5,0.683512,0.316488,0.252440,0.003712,0.003724,9.035353,0.469129,True,67.783937,...,"[[-0.553408145904541, -0.4841879606246948, -0....","[[-0.553408145904541, -0.4841879606246948, -0....","[-0.7032347917556763, -0.717244565486908, -0.7...","[-0.6050304770469666, -0.5582912564277649, -0....",72.90505,-54.083584,-5.213131,9.645178,-4.659138,80.822429
1,10,0.040219,0.459781,0.319781,0.003705,0.003716,1.462513,0.414073,True,87.504347,...,"[[-0.2907896637916565, -0.6671050190925598, 0....","[[-0.2907896637916565, -0.6671050190925598, 0....","[-3.4210522174835205, -2.8223679065704346, -2....","[0.22236865758895874, -0.6671053171157837, 0.1...",113.219742,-75.485542,-5.212150,9.651709,-4.626563,106.725224
2,12,0.211323,0.538677,0.244685,0.009452,0.006355,1.067892,0.375728,True,315.537384,...,"[[-2.568450689315796, -2.6996071338653564, -2....","[[-2.568450689315796, -2.6996071338653564, -2....","[-3.543874502182007, -3.445823907852173, -3.68...","[-4.335927486419678, -4.473454475402832, -4.59...",357.826836,-241.995804,-5.208999,9.663038,-4.576552,144.311363
3,14,0.417366,0.482634,0.212634,0.009032,0.009032,1.024055,0.799226,True,81.262344,...,"[[-1.9443069696426392, -1.9370405673980713, -2...","[[-1.9443069696426392, -1.9370405673980713, -2...","[-2.2571959495544434, -2.3649723529815674, -2....","[-3.0005319118499756, -2.9332122802734375, -2....",92.266466,-69.073189,-5.204783,9.677403,-4.527564,180.619638
4,16,0.092941,0.657059,0.547401,0.004407,0.004447,2.609372,0.361993,True,82.812578,...,"[[0.39511945843696594, 0.32530951499938965, 0....","[[0.39511945843696594, 0.32530951499938965, 0....","[0.3434615731239319, 0.3364798426628113, 0.106...","[-0.0404890701174736, 0.15357960760593414, 0.3...",79.825226,-66.138458,-5.202509,9.687469,-4.504714,199.498275


In [8]:
session_assets = pd.read_csv('/root/capsule/code/data_management/session_assets.csv')
session_list = session_assets['session_id'].tolist()
session_list

['behavior_792292_2025-08-13_14-47-17',
 'behavior_792292_2025-08-15_14-15-47',
 'behavior_796387_2025-09-09_13-37-29',
 'behavior_792288_2025-08-26_11-41-18',
 'behavior_792288_2025-08-27_12-09-38',
 'behavior_791174_2025-09-05_14-06-53',
 'behavior_808655_2025-09-16_10-53-22',
 'behavior_808655_2025-09-18_13-56-51',
 'behavior_808655_2025-09-19_15-00-30',
 'behavior_808650_2025-09-23_14-49-57',
 'behavior_808650_2025-09-24_12-38-00',
 'behavior_808650_2025-09-25_13-35-32',
 'behavior_808650_2025-09-26_11-51-57',
 'behavior_814511_2025-10-16_13-44-32',
 'behavior_814515_2025-10-22_13-03-21',
 'behavior_814515_2025-10-23_14-41-04',
 'behavior_814515_2025-10-24_13-22-07',
 'behavior_810888_2025-10-28_11-51-06',
 'behavior_810888_2025-10-31_12-39-56',
 'behavior_826160_2025-12-19_11-14-24',
 'behavior_826159_2026-01-22_12-58-47',
 'behavior_826159_2026-01-23_13-34-11',
 'behavior_826159_2026-01-26_15-15-12',
 'behavior_826159_2026-01-27_08-55-48',
 'behavior_826164_2026-01-27_11-14-13',


In [10]:
#from joblib import Parallel, delayed
session_list = [
#     'behavior_792292_2025-08-13_14-47-17',
#  'behavior_792292_2025-08-15_14-15-47',
#  'behavior_796387_2025-09-09_13-37-29',
#  'behavior_792288_2025-08-26_11-41-18',
#  'behavior_792288_2025-08-27_12-09-38',
#  'behavior_791174_2025-09-05_14-06-53',
#  'behavior_808655_2025-09-16_10-53-22',
#  'behavior_808655_2025-09-18_13-56-51',
#  'behavior_808655_2025-09-19_15-00-30',
#  'behavior_808650_2025-09-23_14-49-57',
#  'behavior_808650_2025-09-24_12-38-00',
#  'behavior_808650_2025-09-25_13-35-32',
#  'behavior_808650_2025-09-26_11-51-57',
#  'behavior_814511_2025-10-16_13-44-32',
#  'behavior_814515_2025-10-22_13-03-21',
#  'behavior_814515_2025-10-23_14-41-04',
#  'behavior_814515_2025-10-24_13-22-07',
#  'behavior_810888_2025-10-28_11-51-06',
#  'behavior_810888_2025-10-31_12-39-56',
#  'behavior_826160_2025-12-19_11-14-24',
#  'behavior_826159_2026-01-22_12-58-47',
#  'behavior_826159_2026-01-23_13-34-11',
#  'behavior_826159_2026-01-26_15-15-12',
#  'behavior_826159_2026-01-27_08-55-48',
#  'behavior_826164_2026-01-27_11-14-13',
#  'behavior_826164_2026-01-28_11-12-55',
#  'behavior_826164_2026-01-29_11-09-03',
#  'behavior_826164_2026-01-30_11-12-30',
#  'behavior_835444_2026-02-17_13-56-45',
#  'behavior_835444_2026-02-18_13-01-55',
#  'behavior_835444_2026-02-19_13-08-36',
#  'behavior_835444_2026-02-20_12-12-35',
#  'behavior_835451_2026-02-25_13-19-37',
#  'behavior_835451_2026-02-27_14-19-03', #re_filtered waveforms and opto summaries up to here
                # 'behavior_838332_2026-03-10_13-23-52',
                # 'behavior_838332_2026-03-11_13-01-00',
                # 'behavior_838332_2026-03-12_13-31-08',
                # 'behavior_838332_2026-03-13_12-29-22',
                # 'behavior_841596_2026-03-24_13-06-50',
                # 'behavior_841596_2026-03-25_14-10-54',
                'behavior_841596_2026-03-26_14-49-42',
                'behavior_841596_2026-03-27_13-57-43',
                'behavior_841861_2026-04-02_13-12-47',
                'behavior_841861_2026-04-03_13-18-22',
                'behavior_841859_2026-04-07_08-53-01',
                'behavior_841859_2026-04-08_11-03-18',
                'behavior_841859_2026-04-09_09-36-29',
                'behavior_841859_2026-04-10_11-47-55',
                'behavior_843661_2026-04-28_09-49-53',
                'behavior_843661_2026-04-29_08-19-05',
                'behavior_843661_2026-04-30_08-46-20',
                'behavior_843661_2026-05-01_09-00-46',
                'behavior_841598_2026-04-28_12-23-33',
                'behavior_841598_2026-04-29_12-52-23',
                'behavior_841598_2026-04-30_13-45-07',
                'behavior_841598_2026-05-01_14-15-09',
                'behavior_843660_2026-05-05_10-15-53',
                'behavior_843660_2026-05-06_09-15-58',
                'behavior_843660_2026-05-07_09-11-04',
                'behavior_843660_2026-05-08_08-35-38',
                'behavior_848869_2026-05-12_08-51-54',
                'behavior_848869_2026-05-14_08-45-55',
                'behavior_848869_2026-05-15_08-59-55',
                'behavior_841599_2026-05-12_13-01-54',
                'behavior_841599_2026-05-13_13-19-27',
                'behavior_841599_2026-05-14_13-27-52',
                'behavior_841599_2026-05-15_12-57-50',
                'behavior_844917_2026-06-02_11-08-15',
                'behavior_844917_2026-06-03_12-56-55',
                'behavior_844917_2026-06-04_12-30-01',
                'behavior_844917_2026-06-05_12-26-47',
                'behavior_844920_2026-06-10_11-30-14',
                'behavior_844920_2026-06-11_11-27-15',
                'behavior_844920_2026-06-12_11-03-09'
                ]

#session = 'behavior_841596_2026-03-27_13-57-43'

data_type = 'raw'
target='soma'
def process(session, data_type='raw',target='soma'):
    # plt.close('all')
    # session_dir = session_dirs(session)
    #print(session_dir[f'curated_dir_{data_type}'])
    # if session_dir[f'curated_dir_{data_type}'] is None:
    #     return None
    print(f'{session} start')
    # beh_and_time_alignment(session) # plot behavior, check alignment between behavior, sound, and ephys
    # plt.close('all')
    # ephys_opto_preprocessing(session, data_type, target) 
    # redo spiketimes if not aligned, make opto_table (laser time in nidaq); 
    # generate a pkl file with opto_resp/lat to each condition
    plt.close('all')
    # ephys_opto_crosscorr(session, data_type)
    # separate session into opto/beh, cal cross-correlation
    #plt.close('all')
    #opto_wf_preprocessing(session, data_type, target, load_sorting_analyzer=False,unit_filter=False)
    # recal wf based on conditions. (pre/post, power), cal similarity, euclidean distance, correlation with spont
    # pay attention to difference early and late in the session
 
    # unit_tbl = get_unit_tbl(session, data_type, summary=True)
    # if unit_tbl is not None:
    #     if 'peak_waveform_raw_fake_aligned' in unit_tbl.columns:
    #         print(f'Session {session} already processed, skip.')
    #     else:
    #         re_filter_opto_waveforms(session, data_type, opto_only=True, load_sorting_analyzer=True)

    opto_table = opto_plotting_session(session, data_type, target, target_unit_ids=None, plot=False, resp_thresh=0.3, lat_thresh=0.025, save=True)
    # unit_list = opto_table[opto_table['opto_pass'] & opto_table['default_qc']].unit_id.tolist()
    # opto_table_pass = opto_plotting_session(session, 'raw', 'soma', target_unit_ids=unit_list, plot=True, resp_thresh=0.3, lat_thresh=0.025, save=True)
    
    #cal_opto_sigs(session, data_type)

    #plot_session_opto_drift(session, data_type, update_csv=True)
    print(f'{session} stop')
        
 
#Parallel(n_jobs=4)(delayed(process)(session, data_type = data_type) for session in session_list)
for session in session_list:
    try:
        process(session, data_type = 'raw', target='soma')
    except:
        print(f'Failed to process {session}')
#process(session, data_type= data_type)


behavior_841596_2026-03-26_14-49-42 start
122 out of 302 units pass quality control
9 out of 302 units pass quality control and opto tagging
behavior_841596_2026-03-26_14-49-42 stop
behavior_841596_2026-03-27_13-57-43 start
142 out of 288 units pass quality control
10 out of 288 units pass quality control and opto tagging
behavior_841596_2026-03-27_13-57-43 stop
behavior_841861_2026-04-02_13-12-47 start
54 out of 187 units pass quality control
9 out of 187 units pass quality control and opto tagging
behavior_841861_2026-04-02_13-12-47 stop
behavior_841861_2026-04-03_13-18-22 start
16 out of 85 units pass quality control
14 out of 85 units pass quality control and opto tagging
behavior_841861_2026-04-03_13-18-22 stop
behavior_841859_2026-04-07_08-53-01 start
155 out of 392 units pass quality control
16 out of 392 units pass quality control and opto tagging
behavior_841859_2026-04-07_08-53-01 stop
behavior_841859_2026-04-08_11-03-18 start
66 out of 295 units pass quality control
7 out of

In [ ]:
from opto_tagging_summary_DRN import opto_summary_DRN

data_type = 'raw'
target = 'soma'

session_list = [
                'behavior_841596_2026-03-24_13-06-50',
                'behavior_841596_2026-03-25_14-10-54',
                'behavior_841596_2026-03-26_14-49-42',
                'behavior_841596_2026-03-27_13-57-43',
                'behavior_841861_2026-04-02_13-12-47',
                'behavior_841861_2026-04-03_13-18-22',
                'behavior_841859_2026-04-07_08-53-01',
                'behavior_841859_2026-04-08_11-03-18',
                'behavior_841859_2026-04-09_09-36-29',
                'behavior_841859_2026-04-10_11-47-55',
                'behavior_843661_2026-04-28_09-49-53',
                'behavior_843661_2026-04-29_08-19-05',
                'behavior_843661_2026-04-30_08-46-20',
                'behavior_843661_2026-05-01_09-00-46',
                'behavior_841598_2026-04-28_12-23-33',
                'behavior_841598_2026-04-29_12-52-23',
                'behavior_841598_2026-04-30_13-45-07',
                'behavior_841598_2026-05-01_14-15-09',
                'behavior_843660_2026-05-05_10-15-53',
                'behavior_843660_2026-05-06_09-15-58',
                'behavior_843660_2026-05-07_09-11-04',
                'behavior_843660_2026-05-08_08-35-38',
                'behavior_848869_2026-05-12_08-51-54',
                'behavior_848869_2026-05-14_08-45-55',
                'behavior_848869_2026-05-15_08-59-55',
                'behavior_841599_2026-05-12_13-01-54',
                'behavior_841599_2026-05-13_13-19-27',
                'behavior_841599_2026-05-14_13-27-52',
                'behavior_841599_2026-05-15_12-57-50',
                'behavior_844917_2026-06-02_11-08-15',
                'behavior_844917_2026-06-03_12-56-55',
                'behavior_844917_2026-06-04_12-30-01',
                'behavior_844917_2026-06-05_12-26-47',
                'behavior_844920_2026-06-10_11-30-14',
                'behavior_844920_2026-06-11_11-27-15',
                'behavior_844920_2026-06-12_11-03-09'
                ]

for session in session_list:
    unit_tbl = get_unit_tbl(session, data_type, summary=True)
    if unit_tbl is not None:
        if 'corr_max_p' in unit_tbl.columns:
            print(f'Session {session} already processed, skip.')
        else:
            try:
                opto_summary_DRN(session, data_type, target, save=True)
                print(f'{session} opto summary done')
            except:
                print(f'Failed to process {session}')


In [12]:
session = 'behavior_810888_2025-10-31_12-39-56'
opto_table = opto_plotting_session(session, 'raw', 'soma', target_unit_ids=None, plot=False, resp_thresh=0.3, lat_thresh=0.02, save=True)
unit_list = opto_table[opto_table['opto_pass'] & opto_table['default_qc']].unit_id.tolist()

KeyboardInterrupt: 

In [8]:
opto_table = opto_plotting_session(session, 'raw', 'soma', target_unit_ids=unit_list, plot=True, resp_thresh=0.3, lat_thresh=0.02, save=True)

In [4]:
data_type = 'curated'
for session in session_list[-9:]:
    print(f'Checking {session}')
    session_dir = session_dirs(session)
    if os.path.exists(session_dir['sorted_dir_curated']):
        try:
            process(session, data_type = data_type)
            print(f'Finished {session}')
            plt.close('all')
        except:
            print(f'Failed to process {session}')

    # if get_unit_tbl(session, 'curated', summary=False) is None:
    #     print('No curated unit table found.')
    #     if get_unit_tbl(session, 'raw', summary=False) is None:
    #         print('No raw unit table found, processing session')
    #         process(session, data_type='raw') 
    # elif  session_dir['curated_dir_raw'] is not None:
    #     print('Raw')
    #     process(session, data_type='raw')
# process(session_list[-7])
        
 

Checking behavior_784806_2025-06-17_14-59-23
Single experiment found: experiment1, recording1
Single experiment found: experiment1, recording1
/root/capsule/data/behavior_784806_2025-06-17_14-59-23_sorted_curated/curated/experiment1_Record Node 101#Neuropix-PXI-100.ProbeA_recording1
behavior_784806_2025-06-17_14-59-23 start
behavior_784806_2025-06-17_14-59-23
Single experiment found: experiment1, recording1
Single experiment found: experiment1, recording1
Single experiment found: experiment1, recording1
Single experiment found: experiment1, recording1


compute_waveforms (workers: 5 processes):   0%|          | 0/3141 [00:00<?, ?it/s]

Single experiment found: experiment1, recording1
Single experiment found: experiment1, recording1
Single experiment found: experiment1, recording1


compute_waveforms (workers: 5 processes):   0%|          | 0/4163 [00:00<?, ?it/s]

Single experiment found: experiment1, recording1
126 out of 362 units pass quality control


Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x7f0c6408dd20>>
Traceback (most recent call last):
  File "/opt/conda/lib/python3.10/site-packages/ipykernel/ipkernel.py", line 781, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(
KeyboardInterrupt: 


In [ ]:
opto_plotting_session('behavior_784806_2025-06-17_14-59-23', data_type, target)

In [3]:
process('behavior_784806_2025-06-17_14-59-23', data_type='raw')

Single experiment found: experiment1, recording1
There is no nwb file in the raw directory.
None
behavior_751766_2025-02-15_12-08-11 start
behavior_751766_2025-02-15_12-08-11
Single experiment found: experiment1, recording1
There is no nwb file in the raw directory.


behavior_751766_2025-02-15_12-08-11 stop


In [8]:
session_dirs('behavior_744779_2024-11-15_12-54-11')

Single experiment found: experiment1, recording1


{'aniID': '744779',
 'raw_id': '744779_2024-11-15_12-54-11',
 'datetime': datetime.datetime(2024, 11, 15, 12, 54, 11),
 'raw_dir': '/root/capsule/data/behavior_744779_2024-11-15_12-54-11_raw_data',
 'raw_rec': '/root/capsule/data/behavior_744779_2024-11-15_12-54-11_raw_data/ecephys/ecephys_compressed/experiment1_Record Node 104#Neuropix-PXI-100.ProbeA.zarr',
 'session_dir': '/root/capsule/data/behavior_744779_2024-11-15_12-54-11_raw_data/ecephys/ecephys_clipped',
 'session_dir_raw': '/root/capsule/data/behavior_744779_2024-11-15_12-54-11_raw_data/ecephys/ecephys_compressed',
 'processed_dir': '/root/capsule/scratch/744779/behavior_744779_2024-11-15_12-54-11',
 'alignment_dir': '/root/capsule/scratch/744779/behavior_744779_2024-11-15_12-54-11/alignment',
 'beh_fig_dir': '/root/capsule/scratch/744779/behavior_744779_2024-11-15_12-54-11/behavior',
 'ephys_dir_raw': '/root/capsule/scratch/744779/behavior_744779_2024-11-15_12-54-11/ephys/raw',
 'ephys_processed_dir_raw': '/root/capsule/scra

In [9]:
session_list[61:]

['behavior_751181_2025-02-25_12-12-35',
 'behavior_751181_2025-02-26_11-51-19',
 'behavior_751181_2025-02-27_11-24-47',
 'behavior_754897_2025-03-11_12-07-41',
 'behavior_754897_2025-03-12_12-23-15',
 'behavior_754897_2025-03-13_11-20-42',
 'behavior_754897_2025-03-14_11-28-53',
 'behavior_754897_2025-03-15_11-32-18',
 'behavior_758018_2025-03-19_11-16-44',
 'behavior_758018_2025-03-20_11-53-05',
 'behavior_758018_2025-03-21_11-00-34',
 'behavior_752014_2025-03-25_12-09-20',
 'behavior_752014_2025-03-26_11-18-57',
 'behavior_752014_2025-03-27_12-03-59',
 'behavior_752014_2025-03-28_11-04-59',
 'behavior_761038_2025-04-15_10-25-11',
 'behavior_761038_2025-04-16_10-39-10',
 'behavior_761038_2025-04-17_11-03-16',
 'behavior_761038_2025-04-18_12-37-39',
 'ecephys_763360_2025-04-15_12-16-29',
 'ecephys_763360_2025-04-16_13-29-55',
 'behavior_782394_2025-04-22_10-53-28',
 'behavior_782394_2025-04-23_10-51-17',
 'behavior_782394_2025-04-24_12-07-34',
 'behavior_782394_2025-04-25_11-13-21',
 '

In [25]:
rec = si.load('/root/capsule/data/ecephys_713854_2024-03-05_13-31-20_raw_data/ecephys_compressed/experiment1_Record Node 104#Neuropix-PXI-100.ProbeA.zarr')

OSError: [Errno 5] Input/output error

In [10]:
len(os.listdir(os.path.join(session_dir['opto_dir_raw'], 'figures')))

[]

In [12]:
tbl = get_unit_tbl('behavior_716325_2024-05-30_11-33-46', 'raw')

Selected experiment1 recording1, length:3442.18
No unit table found for behavior_716325_2024-05-30_11-33-46 in raw data.


In [14]:
tbl is None

True

In [6]:
session = 'behavior_791691_2025-06-24_13-21-29'
process(session)
# session_dir = session_dirs(session)

NameError: name 'process' is not defined

In [8]:
session_dir['nwb_dir_raw']

'/root/capsule/data/behavior_791691_2025-06-24_13-21-29_sorted/nwb/behavior_791691_2025-06-24_13-21-26_experiment2_recording1.nwb'

In [4]:
session = 'behavior_791691_2025-06-24_13-21-29'
session_dir = session_dirs(session)

Selected experiment1 recording1, length:4862.23


In [4]:
session_dir['curated_dir_raw']

'/root/capsule/data/behavior_782394_2025-04-22_10-53-28_sorted/curated/experiment1_Record Node 104#Neuropix-PXI-100.ProbeA_recording1'

In [ ]:
os.path.exists(session_dir['sorted_dir_raw'])

True

In [4]:
from open_ephys.analysis import Session

In [5]:
rec_file = '/root/capsule/data/behavior_791691_2025-06-24_13-21-29_raw_data/ecephys/ecephys_clipped'
rec_clipped = Session(rec_file)

In [29]:
stream_info = get_stream_info("/root/capsule/data/ecephys_713854_2024-03-05_13-31-20_raw_data/ecephys_clipped")

In [30]:
stream_info

,Record Node,Rec Idx,Exp Idx,Stream,Duration (s),Channels
0,104,0,0,ProbeA,0.003333,384
1,104,0,0,PXIe-6341,0.003333,8


In [19]:
rec_clipped.recordnodes[0].recordings[0].continuous[0].timestamps[-1]-rec_clipped.recordnodes[0].recordings[0].continuous[0].timestamps[0]

np.float64(5118.98209682107)

In [22]:
rec_clipped.recordnodes[0]

AttributeError: 'RecordNode' object has no attribute 'metadata'

In [ ]:
# make a df from all_Tm, columns correspond to focus regressors
 